# 🛡️ Optiv Consulting — Case Study-2: Text Analysis and PII Detection
### Client: Cadence Financial Services (New York)
**Objective**: Extract text from varied evidence artifacts (DOCX, PPTX, PDF, TXT), detect and classify embedded PII with near-100% recall, preserve >= 80% structural integrity, and sanitize documents before downstream LLM processing.

In [ ]:
# 1. Setup & Imports
import os
import pandas as pd
from extractors.unified_extractor import UnifiedExtractor
from detector.pii_engine import PIIEngine
from redactor.redaction_engine import RedactionEngine
from audit.risk_assessment import RiskAssessmentEngine

print("✅ Optiv PII Detection Pipeline Loaded Successfully!")

## 2. Ingest Multi-Format Evidence Artifacts
Let's inspect one of Cadence's confidential evidence artifacts (e.g., Commercial Credit Risk PDF or Wealth Review PPTX).

In [ ]:
sample_file = os.path.join("samples", "cadence_credit_policy_evaluation.pdf")

# Step 1: Extraction & Structure Integrity Evaluation
doc = UnifiedExtractor.extract_document(sample_file)

print(f"Document: {doc.file_name}")
print(f"Format: {doc.file_type.upper()}")
print(f"Total Words: {doc.total_words}")
print(f"Structural Blocks: {doc.total_blocks}")
print(f"Structural Retention Score: {doc.metadata.get('structural_retention_pct')} (Requirement >= 80%: {doc.metadata.get('structure_valid')})")

## 3. Multi-Tier Hybrid PII Detection Engine
Combines Microsoft Presidio NER (spaCy), Deterministic Checksum & RegEx Validators (Luhn CC, ABA routing, SSN, PAN, Aadhaar), and Cadence Contextual Heuristics.

In [ ]:
# Step 2: Detect PII
entities = PIIEngine.detect_in_document(doc)

print(f"Total Sensitive Entities Detected: {len(entities)}")

# Display summary table of detected entities
preview_records = []
for e in entities[:10]:
    preview_records.append({
        "Entity Type": e.entity_type,
        "Category": e.category,
        "Severity": e.severity,
        "Detected Value": e.value,
        "Confidence": f"{int(e.confidence*100)}%",
        "Source": e.source
    })

pd.DataFrame(preview_records)

## 4. Semantic Redaction & Reversible Token Vault

In [ ]:
# Step 3: Redact and create reversible token vault
redacted_doc, vault = RedactionEngine.redact_document(doc, entities)

print(f"Unique Surrogate Tokens Created: {len(vault.reverse_map)}")
print("\n--- Sample Token Map (for authorized enclave de-anonymization) ---")
sample_tokens = list(vault.reverse_map.items())[:5]
for tok, data in sample_tokens:
    print(f"{tok} -> {data['original_value']} ({data['entity_type']})")

## 5. Downstream AI Agent Sanitized Prompt Generation
Confirms 0.0% residual PII before payload is dispatched to the LLM agent.

In [ ]:
# Step 4: Generate LLM Ready Payload
llm_payload = RedactionEngine.generate_llm_payload(redacted_doc)

print(f"Status: {llm_payload['pii_leakage_risk']}")
print(f"Structural Retention: {llm_payload['structural_retention_score']}")
print("\n--- Sanitized Prompt Preview Sent to Cadence LLM Agent ---\n")
print(llm_payload['sanitized_payload'][:600] + "\n...")

## 6. Cadence Regulatory Compliance & Exposure Audit Report

In [ ]:
# Step 5: Risk Assessment
audit_report = RiskAssessmentEngine.generate_audit_report(doc, entities)

print(f"Cadence Exposure Score: {audit_report.exposure_score}/100")
print(f"Risk Level: {audit_report.exposure_level}")
print(f"PII Density: {audit_report.pii_density_per_100w} instances / 100 words")
print("\nCategory Breakdown:")
for cat, count in audit_report.breakdown_by_category.items():
    print(f"  - {cat}: {count}")

print("\nRegulatory Impacts:")
for r in audit_report.regulatory_impacts:
    print(f"  ⚠️ [{r['framework']}] {r['mandate']}: {r['risk']}")